In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 21:07:20.521884: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 21:07:21.307078: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 21:07:22,422 [DEBUG] [Rain] Rain is initialized
2023-07-04 21:07:22,423 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 21:07:22,424 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/coord/
2023-07-04 21:07:22,425 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 21:07:22,426 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 21:07:22,428 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 21:07:22,429 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 21:07:22,431 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 21:07:22,437 [DEBUG] [Rain] Creating workers
2023-07-04 21:07:22,445 [INFO] [Provisioner] provisioner is serving
2023-07-04 21:07:22,446 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 21:07:22,448 [INFO] [Coordinator] coordinator is serving
2023-07-04 21:07:22,449 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 21:07:22,453 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 21:07:22,455 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 21:07:22,456 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 21:07:22,457 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 21:07:22,459 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 21:07:22,461 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 21:07:22,463 [INFO] [Worker_

123/157 [======================>.......] - ETA: 0s - loss: 0.7884 - accuracy: 0.7512

2023-07-04 21:07:42,662 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:07:42,665 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider
157/157 [==============================] - 3s 10ms/step - loss: 0.7063 - accuracy: 0.7775


2023-07-04 21:07:42,894 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:07:42,896 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

2023-07-04 21:07:51,315 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:07:51,317 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
157/157 [==============================] - 3s 7ms/step - loss: 0.2509 - accuracy: 0.9260


2023-07-04 21:07:51,404 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:07:51,406 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-04 21:07:51,537 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 21:07:51,548 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-04 21:07:51,588 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.
2023-07-04 21:07:51,604 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 21:07:51,615 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-04 21:07:51,640 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLe

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.1382 - accuracy: 0.9585

Test accuracy: 95.9%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 21:07:52,047 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 21:07:52,050 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 21:07:52,052 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 21:07:52,054 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 21:07:52,055 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 21:07:52,057 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 21:07:52,060 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 21:07:52,

153/157 [============================>.] - ETA: 0s - loss: 0.2043 - accuracy: 0.9392

2023-07-04 21:08:13,007 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to divider
156/157 [============================>.] - ETA: 0s - loss: 0.2047 - accuracy: 0.9395

DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:08:13,019 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

sending data to divider


2023-07-04 21:08:13,261 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 21:08:13,270 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
2023-07-04 21:08:13,891 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 21:08:13,893 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-04 21:08:13,995 [DEBUG] [DividerAmbassador] Download

sending data to divider


2023-07-04 21:08:14,401 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
DEBUG:DividerAmbassador:divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 21:08:14,403 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker2
DEBUG:DividerAmbassador:divider begins executing iteration2 for worker2
2023-07-04 21:08:14,405 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-04 21:08:14,405 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
INFO:Worker_50152:Running the worker with id: 2 on iteration: 2
2023-07-04 21:08:14,410 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 21:08:14,410 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
DEBUG:DividerAmbassador:divider received: File downloaded successful

146/157 [==========================>...] - ETA: 0s - loss: 0.1811 - accuracy: 0.9450

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1069 - accuracy: 0.9676

Test accuracy: 96.8%
